# 00-Initialization

In [1]:
import os
import torch
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 01-Loading Data

In [11]:
from datasets import load_dataset
dataset_halu_qa = load_dataset('/YOUR_PATH/data/notrichardren___halu_eval/qa')
dataset_trivia = load_dataset('/YOUR_PATH/data/RUC-NLPIR___flash_rag_datasets/triviaqa')
dataset_nq = load_dataset('/YOUR_PATH/data/RUC-NLPIR___flash_rag_datasets/nq')

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

# 02-Loading Model

In [3]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM,AutoTokenizer

model_name = ["llama3","qwen2.5-7B",'qwen2.5-14B']
model_path = ["/YOUR_PATH/modelscope/LLM-Research/Meta-Llama-3___1-8B-Instruct",
              "/YOUR_PATH/modelscope/Qwen/Qwen2.5-7B-Instruct",
              "/YOUR_PATH/modelscope/models/Qwen/Qwen2-5-14B-Instruct"
              ]
m = 0
tokenizer = AutoTokenizer.from_pretrained(f"{model_path[m]}")
model = AutoModelForCausalLM.from_pretrained(f"{model_path[m]}",device_map='auto',torch_dtype=torch.float32,).eval()

SenSimModel = SentenceTransformer('/YOUR_PATH/modelscope/models--sentence-transformers--nli-roberta-large/snapshots/f8af28fc8246358d69bfe9377837798ecafafdab/')


/home/alva/anaconda3/envs/flexrag/lib/python3.11/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

No sentence-transformers model found with name /YOUR_PATH/modelscope/models--sentence-transformers--nli-roberta-large/snapshots/f8af28fc8246358d69bfe9377837798ecafafdab/. Creating a new one with mean pooling.


# 03-Funcitions

In [4]:
import os
import random
import numpy as np
from sentence_transformers import util
from rouge_score import rouge_scorer


rougeEvaluator = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def seed_everything(seed: int):
    if seed is None:
        return
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def softmax(x):
    exp_values = np.exp(x - np.max(x))
    return exp_values / np.sum(exp_values)

def get_perplexity_score(scores):
    perplexity = 0.0
    for logits in scores:
        conf = torch.max(logits.softmax(1)).cpu().item()
        perplexity += np.log(conf)
    perplexity = -1.0 * perplexity/len(scores)
    return perplexity

def get_energy_score(scores):
    avg_energy = 0.0
    for logits in scores:
        # logits[logits<-100]=-100
        # logits[logits>100]=100
        # logits = logits/100
        # energy = -torch.log((torch.exp(logits)).sum()).item()
        energy = - torch.logsumexp(logits[0], dim=0, keepdim=False).item()
        avg_energy += energy
    avg_energy = avg_energy/len(scores)
    return avg_energy

def getSentenceSimilarity(generations, answers, SenSimModel):
    gen_embeddings = SenSimModel.encode(generations)
    ans_embeddings = SenSimModel.encode(answers)
    similarity = util.cos_sim(gen_embeddings, ans_embeddings)
    return similarity.item()

def get_lenghthNormalized_entropy(batch_scores, num_tokens):
    seq_entropy = np.zeros(len(num_tokens))
    for ind1, logits in enumerate(batch_scores):
        for ind2, seq_logits in enumerate(logits):
            if ind1 < num_tokens[ind2]:
                conf, _ = torch.max(seq_logits.softmax(0), dim=0)
                seq_entropy[ind2] = seq_entropy[ind2] + np.log(conf.cpu().numpy())
    normalized_entropy = 0
    for ind, entropy in enumerate(seq_entropy):
        normalized_entropy += entropy/num_tokens[ind]
    normalized_entropy = -1.0* normalized_entropy/len(num_tokens)
    return normalized_entropy

def getEigenIndicator_v0(hidden_states, num_tokens):
    alpha = 1e-3
    selected_layer = int(len(hidden_states[0])/2)
    # selected_layer = -1
    if len(hidden_states)<1:
        return 0, "None"
    last_embeddings = torch.zeros(hidden_states[0][-1].shape[0], hidden_states[0][-1].shape[2]).to("cuda")
    for ind in range(hidden_states[0][-1].shape[0]):
        last_embeddings[ind,:] = hidden_states[num_tokens[ind]-1][selected_layer][ind,0,:]
    CovMatrix = torch.cov(last_embeddings).cpu().numpy().astype(float)
    u, s, vT = np.linalg.svd(CovMatrix+alpha*np.eye(CovMatrix.shape[0]))
    #====================
    s = np.clip(s, a_min=1e-10, a_max=None)
    eigenIndicator = np.mean(np.log10(s))
    return eigenIndicator, s

def getLexicalSim(generated_texts):
    LexicalSim = 0
    for i in range(len(generated_texts)):
        for j in range(len(generated_texts)):
            if j<=i:
                continue
            LexicalSim += getRouge(rougeEvaluator, generated_texts[i], generated_texts[j])
    LexicalSim = LexicalSim/(len(generated_texts)*(len(generated_texts)-1)/2)
    return LexicalSim

def getRouge(rouge, generations, answers):
    # results = rouge.compute(predictions=[generations], references=[answers], use_aggregator=False)
    results = rouge.score(target = answers, prediction = generations)
    RoughL = results["rougeL"].fmeasure  #fmeasure/recall/precision
    return RoughL

def get_num_tokens(generation):  # generation: num_seq x max(num_tokens)
    num_tokens = []
    for ids in generation:
        count = 0
        for id in ids:
            if id>2:
                count+=1
        num_tokens.append(count+1)
    return num_tokens

def eigen_confidence(λ):
    λ = np.asarray(λ, dtype=float)
    λ /= λ.sum()
    return λ[0]

# 04-Hallucination Detection

In [5]:
from tqdm import tqdm
import pandas as pd

dataset_names = ['nq', 'trivia', 'halu_qa']
dataset_name = dataset_names[0]
raw_ds = dataset_nq
dataset = raw_ds['test']

num_gens = 2
c_tf_nums= num_gens
sequences = []


sample_length = 2
for d in tqdm(range(1,sample_length)):

    user_message = [{"role": "system", "content":"You are a helpful AI assistant. Answer the question directly. No extra words and avoid full sentences."},{"role": "user", "content":f"Question: {dataset[d]['question']}."}]
    prompt = tokenizer.apply_chat_template(user_message, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt",add_special_tokens=False).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    output_ids = model.generate(input_ids=input_ids,
                            attention_mask=attention_mask,
                            num_beams=1,
                            do_sample=False,
                            output_hidden_states = True,
                            return_dict_in_generate=True,
                            output_scores=True,
                            max_new_tokens=512,
                            output_attentions = True
                            )
    output = tokenizer.decode(output_ids[0][0][input_ids.shape[1]:-1],skip_special_tokens=True )

    scores = output_ids.scores[:-1]    #([logits],[logits],[logits])
    perplexity = get_perplexity_score(scores)
    energy_score = get_energy_score(scores)

    input_length = input_ids.shape[1]
    most_likely_generations = output_ids.sequences.cpu()[0, input_length:]

    torch.cuda.empty_cache()

#===============================================================================================================
    generations = []

    dict_outputs =  model.generate(input_ids=input_ids,attention_mask=attention_mask,num_beams=1, num_return_sequences=num_gens,do_sample=True,
                                    top_p=0.99, top_k=5,temperature=0.5,max_new_tokens=512,repetition_penalty=1.2,
                                    output_hidden_states = True, return_dict_in_generate=True, output_scores=True)

    generation = dict_outputs.sequences[:, input_length:].cpu()
    generations.append(generation)
    # num_tokens = get_num_tokens(generation)

    stop_token = tokenizer.eos_token_id # 128009
    num_tokens = (generation == stop_token).int().argmax(dim=1).tolist()

    scores = dict_outputs.scores[:-1]

    predictive_entropy = get_lenghthNormalized_entropy(scores, num_tokens)
    hidden_states = dict_outputs.hidden_states[:-1]
    eigenIndicator, eigenValue = getEigenIndicator_v0(hidden_states, num_tokens)


    generations = torch.nested.nested_tensor(generations).to_padded_tensor(tokenizer.eos_token_id)
    generations = generations.reshape(-1, generations.shape[-1])
    best_generated_text = tokenizer.decode(most_likely_generations, skip_special_tokens=True)
    generated_texts = [tokenizer.decode(_, skip_special_tokens=True) for _ in generations]


    confident = eigen_confidence(eigenValue)

    if dataset_name == 'halu_qa':
        curr_seq = dict(
            prompt=tokenizer.decode(input_ids.cpu()[0], skip_special_tokens=True),
            id = d,
            question=dataset[d]['question'],
            answer = dataset[d]['right_answer'],
        )
    else:
        curr_seq = dict(
        prompt=tokenizer.decode(input_ids.cpu()[0], skip_special_tokens=True),
        id=dataset[d]['id'],
        question=dataset[d]['question'],
        answer = [])
        if dataset_name == 'nq' or dataset_name == "trivia":
            curr_seq['answer'] = dataset[d]['golden_answers']

    curr_seq.update(
        dict(
            most_likely_generation_ids = most_likely_generations,
            generations_ids=generations,
        )
    )
    curr_seq.update(
        dict(
            most_likely_generation=tokenizer.decode(curr_seq['most_likely_generation_ids'], skip_special_tokens=True),
            generations=generated_texts,
        )
    )

    curr_seq.update({
        'perplexity': perplexity,
        'energy': energy_score,
        'entropy':predictive_entropy,
        'eigenIndicator':eigenIndicator,
        'confident':confident,
        'eigen_value':eigenValue,
    })

    sequences.append(curr_seq)
    torch.cuda.empty_cache()
# pd.to_pickle(sequences, f'/home/ljr2025/my_paper/03_condition_perplexity/{model_name[m]}/{dataset_name}_{sample_length}.pkl')

  0%|          | 0/1 [00:00<?, ?it/s]/home/alva/anaconda3/envs/flexrag/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/alva/anaconda3/envs/flexrag/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
Setting `pad

In [6]:
sequences

[{'prompt': 'system\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful AI assistant. Answer the question directly. No extra words and avoid full sentences.user\n\nQuestion: when is the next deadpool movie being released.assistant\n\n',
  'id': 'test_1',
  'question': 'when is the next deadpool movie being released',
  'answer': ['May 18, 2018'],
  'most_likely_generation_ids': tensor([ 29420,  10497,    220,     18,     25,   3297,    220,   2366,     19,
          128009]),
  'generations_ids': tensor([[ 29420,  10497,    220,     18,   4984,   2457,     25,   6841,    220,
               23,     11,    220,   2366,     19,    320,   2078,      8, 128009],
          [ 29420,  10497,    220,     18,   4984,   2457,     25,   6841,    220,
               23,     11,    220,   2366,     19,    320,   2078,      8, 128009]]),
  'most_likely_generation': 'Deadpool 3: May 2024',
  'generations': ['Deadpool 3 release date: November 8, 2024 (US)',
   'Deadp

# 05-Results

In [ ]:

import numpy as np
import io
from eval_func import *
import pickle

model_name = ["llama3","qwen2.5-7B",'qwen2.5-14B']
m = 0
dataset_names = ['nq', 'trivia', 'halu_qa']
dataset_name = dataset_names[0]
file_name = f'/home/ljr2025/my_paper/03_condition_perplexity/{model_name[m]}/{dataset_name}_1000_01.pkl'

f = open(file_name, "rb")
resultDict = pickle.load(f)


In [8]:

Label = []
Score = []
ppl=[]
energy=[]
lne=[]
eig=[]

AIC = []
USE_Roberta = 0
for item in resultDict:
    ansGT = item["answer"]
    generations = item["most_likely_generation"]
    ppl.append(item["perplexity"])
    energy.append(-item["energy"])
    lne.append(-item["entropy"])
    eig.append(-item["eigenIndicator"])
    AIC.append(eigen_confidence(item["eigen_value"]))

    if USE_Roberta:
        if isinstance(ansGT,list):
            ss = []
            for ansGTs in ansGT:
                Score00 = getSentenceSimilarity(generations, ansGTs, SenSimModel)
                ss.append(Score00)
            similarity = max(ss)
        else:
            similarity = getSentenceSimilarity(generations, ansGT, SenSimModel)

        if similarity>0.9:
            Label.append(1)
        else:
            Label.append(0)
        Score.append(similarity)
    else:
        if isinstance(ansGT,list):
            ss = []
            for ansGTs in ansGT:
                gen = generations
                ref = ansGTs
                # lower & split
                gen_tok = gen.lower()
                ref_tok = ref.lower()
                Score00 = getRouge(rougeEvaluator, gen_tok, ref_tok)
                ss.append(Score00)
            rougeScore = max(ss)
        else:
            rougeScore = getRouge(rougeEvaluator, generations.lower(), ansGT.lower())

        if rougeScore>0.5:
            Label.append(1)
        else:
            Label.append(0)
        Score.append(rougeScore)



In [10]:
print(Label)

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 